# Phase 3: Model Bake-off

Goal: confirm whether XGBoost is still the right production model now that Phase 1 expanded the feature set to 105 columns (geographic + DAS). All candidates train on identical splits and we judge on the same held-out test parquet.

**Candidates**

| Model | Task | Notes |
|-------|------|-------|
| XGBoost v2 | both | already trained, loaded as baseline |
| LightGBM 4.x | both | fast histogram gbm |
| CatBoost | both | fed the same one-hot parquet for a fair comparison (native cat features would require a separate preprocessing branch) |
| TabPFN-2.5 | both | transformer foundation model, CPU only on this Mac |
| AutoGluon-Tabular | optional | gated behind `RUN_AUTOGLUON` flag — heavy install |

**Outputs**
- single comparison table with MAE, RMSE, R², AUC, F1, latency
- `models/best_v3.pkl` — winner under a stable name
- decision recorded in the closing markdown cell

In [1]:
import json
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, f1_score, mean_absolute_error, precision_score,
    r2_score, recall_score, roc_auc_score, root_mean_squared_error,
)

warnings.filterwarnings('ignore')

ROOT = Path('..').resolve()
DATA = ROOT / 'data'
MODELS = ROOT / 'models'
FIGS = ROOT / 'figures'
FIGS.mkdir(exist_ok=True)

# Toggle to True only if autogluon-tabular is installed (~2GB)
RUN_AUTOGLUON = False

# TabPFN test-set subsample to keep CPU runtime reasonable.
# Set to None to score the full 5.3K-row test set (slow).
TABPFN_TEST_SAMPLE = 2000

train_df = pd.read_parquet(DATA / 'train.parquet')
val_df   = pd.read_parquet(DATA / 'val.parquet')
test_df  = pd.read_parquet(DATA / 'test.parquet')

TARGET_COLS = ['dim_flag', 'log_net_charge', 'Net Charge Billed Currency',
               'log_base_charge', 'log_misc_charge']
feature_cols = [c for c in train_df.columns if c not in TARGET_COLS]

X_train = train_df[feature_cols]
X_val   = val_df[feature_cols]
X_test  = test_df[feature_cols]

y_train_cls = train_df['dim_flag'].astype(int)
y_val_cls   = val_df['dim_flag'].astype(int)
y_test_cls  = test_df['dim_flag'].astype(int)

y_train_log = train_df['log_net_charge']
y_val_log   = val_df['log_net_charge']
y_test_log  = test_df['log_net_charge']
y_test_dol  = test_df['Net Charge Billed Currency']

print(f'train {X_train.shape}  val {X_val.shape}  test {X_test.shape}')
print(f'features = {len(feature_cols)}')

cls_rows = []
reg_rows = []

train (45508, 105)  val (5688, 105)  test (5689, 105)
features = 105


In [2]:
def eval_cls(name, model, X, y_true):
    t0 = time.perf_counter()
    proba = model.predict_proba(X)
    latency_ms = 1000 * (time.perf_counter() - t0) / len(X)
    if proba.ndim == 2:
        cls_idx = list(model.classes_).index(1) if 1 in list(model.classes_) else 1
        p_pos = proba[:, cls_idx]
    else:
        p_pos = proba
    pred = (p_pos >= 0.5).astype(int)
    return {
        'model': name,
        'accuracy':  accuracy_score(y_true, pred),
        'precision': precision_score(y_true, pred, zero_division=0),
        'recall':    recall_score(y_true, pred),
        'f1':        f1_score(y_true, pred),
        'auc':       roc_auc_score(y_true, p_pos),
        'latency_ms_per_row': latency_ms,
    }

def eval_reg(name, model, X, y_dollars):
    t0 = time.perf_counter()
    log_pred = model.predict(X)
    latency_ms = 1000 * (time.perf_counter() - t0) / len(X)
    dol_pred = np.expm1(log_pred)
    return {
        'model': name,
        'mae_dollars':  mean_absolute_error(y_dollars, dol_pred),
        'rmse_dollars': root_mean_squared_error(y_dollars, dol_pred),
        'r2':           r2_score(y_dollars, dol_pred),
        'latency_ms_per_row': latency_ms,
    }

## XGBoost v2 (reference)

Already trained in Phase 1. Loaded only to occupy a row in the comparison table.

In [3]:
xgb_clf = joblib.load(MODELS / 'xgb_classifier_v2.pkl')
xgb_reg = joblib.load(MODELS / 'xgb_regressor_v2.pkl')

cls_rows.append(eval_cls('XGBoost v2', xgb_clf, X_test, y_test_cls))
reg_rows.append(eval_reg('XGBoost v2', xgb_reg, X_test, y_test_dol))
print(cls_rows[-1])
print(reg_rows[-1])

{'model': 'XGBoost v2', 'accuracy': 0.9973633327474073, 'precision': 0.9956308028399782, 'recall': 0.9961748633879781, 'f1': 0.9959027588090685, 'auc': 0.9997414319233868, 'latency_ms_per_row': 0.004011432939969212}
{'model': 'XGBoost v2', 'mae_dollars': 2.7720459437156566, 'rmse_dollars': 6.932312044692716, 'r2': 0.8883819102921425, 'latency_ms_per_row': 0.006132873439709765}


## LightGBM

In [4]:
import lightgbm as lgb

lgb_clf = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1,
)
lgb_clf.fit(
    X_train, y_train_cls,
    eval_set=[(X_val, y_val_cls)],
    callbacks=[lgb.early_stopping(20, verbose=False), lgb.log_evaluation(0)],
)

lgb_reg = lgb.LGBMRegressor(
    n_estimators=3000, learning_rate=0.03, num_leaves=63, max_depth=10,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, n_jobs=-1, verbose=-1,
)
lgb_reg.fit(
    X_train, y_train_log,
    eval_set=[(X_val, y_val_log)],
    callbacks=[lgb.early_stopping(40, verbose=False), lgb.log_evaluation(0)],
)

cls_rows.append(eval_cls('LightGBM', lgb_clf, X_test, y_test_cls))
reg_rows.append(eval_reg('LightGBM', lgb_reg, X_test, y_test_dol))
print(cls_rows[-1])
print(reg_rows[-1])

{'model': 'LightGBM', 'accuracy': 0.9971875549305678, 'precision': 0.9945474372955289, 'recall': 0.9967213114754099, 'f1': 0.9956331877729258, 'auc': 0.9998749640681, 'latency_ms_per_row': 0.0025483680789702367}
{'model': 'LightGBM', 'mae_dollars': 2.7461433026665443, 'rmse_dollars': 6.954342968897708, 'r2': 0.8876713372265356, 'latency_ms_per_row': 0.04061043962063288}


## CatBoost

Reads the same one-hot encoded parquet. Native-categorical mode would require a separate preprocessing branch; left as future work.

In [5]:
from catboost import CatBoostClassifier, CatBoostRegressor

neg, pos = int((y_train_cls == 0).sum()), int((y_train_cls == 1).sum())
scale_pos_weight = neg / max(pos, 1)

cat_clf = CatBoostClassifier(
    iterations=500, learning_rate=0.05, depth=6,
    scale_pos_weight=scale_pos_weight,
    eval_metric='AUC', random_state=42, verbose=0,
    early_stopping_rounds=20,
)
cat_clf.fit(X_train, y_train_cls, eval_set=(X_val, y_val_cls))

cat_reg = CatBoostRegressor(
    iterations=3000, learning_rate=0.03, depth=10,
    l2_leaf_reg=3.0, random_state=42, verbose=0,
    early_stopping_rounds=40,
)
cat_reg.fit(X_train, y_train_log, eval_set=(X_val, y_val_log))

cls_rows.append(eval_cls('CatBoost', cat_clf, X_test, y_test_cls))
reg_rows.append(eval_reg('CatBoost', cat_reg, X_test, y_test_dol))
print(cls_rows[-1])
print(reg_rows[-1])

{'model': 'CatBoost', 'accuracy': 0.9971875549305678, 'precision': 0.9950873362445415, 'recall': 0.9961748633879781, 'f1': 0.9956308028399782, 'auc': 0.9994371259011295, 'latency_ms_per_row': 0.002787506591632116}
{'model': 'CatBoost', 'mae_dollars': 2.67653360690603, 'rmse_dollars': 6.574420582992598, 'r2': 0.8996093315790421, 'latency_ms_per_row': 0.002446666022391126}


## TabPFN-2.5

Transformer foundation model with in-context learning. CPU-only on this Mac.
Test set subsampled (`TABPFN_TEST_SAMPLE`) for latency; same sample is used for AUC/MAE so the comparison stays fair.

In [6]:
try:
    from tabpfn import TabPFNClassifier, TabPFNRegressor
    TABPFN_OK = True
except Exception as e:
    print(f'TabPFN import failed: {e}')
    TABPFN_OK = False

tab_clf = None
tab_reg = None

if TABPFN_OK:
    rng = np.random.default_rng(42)
    if TABPFN_TEST_SAMPLE and TABPFN_TEST_SAMPLE < len(X_test):
        sample_idx = rng.choice(len(X_test), TABPFN_TEST_SAMPLE, replace=False)
        X_test_t  = X_test.iloc[sample_idx]
        y_test_ct = y_test_cls.iloc[sample_idx]
        y_test_dt = y_test_dol.iloc[sample_idx]
    else:
        X_test_t, y_test_ct, y_test_dt = X_test, y_test_cls, y_test_dol

    if len(X_train) > 10000:
        tr_idx = rng.choice(len(X_train), 10000, replace=False)
        X_tr_t = X_train.iloc[tr_idx]
        y_tr_ct = y_train_cls.iloc[tr_idx]
        y_tr_lt = y_train_log.iloc[tr_idx]
    else:
        X_tr_t, y_tr_ct, y_tr_lt = X_train, y_train_cls, y_train_log

    print(f'TabPFN train rows: {len(X_tr_t)}   test rows: {len(X_test_t)}')

    # TabPFN-v8 requires a license token + downloaded weights on first run.
    # If that flow can't complete (no TABPFN_TOKEN, no interactive terminal,
    # offline) we capture the error and continue the bake-off without it —
    # this is documented in README as a known limitation of v8.
    try:
        tab_clf = TabPFNClassifier(device='cpu', ignore_pretraining_limits=True)
        tab_clf.fit(X_tr_t.values, y_tr_ct.values)
        cls_rows.append(eval_cls('TabPFN-2.5', tab_clf, X_test_t.values, y_test_ct.values))
        print(cls_rows[-1])
    except Exception as e:
        print(f'TabPFN classifier skipped: {type(e).__name__}: {str(e)[:200]}')
        tab_clf = None

    try:
        tab_reg = TabPFNRegressor(device='cpu', ignore_pretraining_limits=True)
        tab_reg.fit(X_tr_t.values, y_tr_lt.values)
        reg_rows.append(eval_reg('TabPFN-2.5', tab_reg, X_test_t.values, y_test_dt.values))
        print(reg_rows[-1])
    except Exception as e:
        print(f'TabPFN regressor skipped: {type(e).__name__}: {str(e)[:200]}')
        tab_reg = None

    TABPFN_OK = tab_clf is not None or tab_reg is not None
else:
    print('Skipping TabPFN (import failed).')

TabPFN train rows: 10000   test rows: 2000


TabPFN classifier skipped: TabPFNLicenseError: TabPFN requires a one-time license acceptance to download
model weights for local inference, but no interactive terminal
is available.

To authenticate in a non-interactive environment:
  1. Open http


TabPFN regressor skipped: TabPFNLicenseError: TabPFN requires a one-time license acceptance to download
model weights for local inference, but no interactive terminal
is available.

To authenticate in a non-interactive environment:
  1. Open http


## AutoGluon-Tabular (optional)

Gated behind `RUN_AUTOGLUON`. Stacks LightGBM, CatBoost, XGBoost, NN, TabPFN automatically — included as an anchor for "what's the best we can do."

In [7]:
if RUN_AUTOGLUON:
    from autogluon.tabular import TabularPredictor

    ag_train = train_df.copy()
    ag_val   = val_df.copy()
    ag_test  = test_df.copy()

    # Classification
    drop_for_cls = [c for c in TARGET_COLS if c != 'dim_flag']
    ag_predictor_cls = TabularPredictor(
        label='dim_flag', eval_metric='roc_auc',
        path=str(MODELS / 'autogluon_cls'),
    ).fit(
        ag_train.drop(columns=drop_for_cls),
        tuning_data=ag_val.drop(columns=drop_for_cls),
        time_limit=1800, presets='medium_quality',
    )
    proba = ag_predictor_cls.predict_proba(ag_test.drop(columns=drop_for_cls + ['dim_flag']))
    p_pos = proba[1].values if 1 in proba.columns else proba.iloc[:, -1].values
    pred = (p_pos >= 0.5).astype(int)
    cls_rows.append({
        'model': 'AutoGluon',
        'accuracy':  accuracy_score(y_test_cls, pred),
        'precision': precision_score(y_test_cls, pred, zero_division=0),
        'recall':    recall_score(y_test_cls, pred),
        'f1':        f1_score(y_test_cls, pred),
        'auc':       roc_auc_score(y_test_cls, p_pos),
        'latency_ms_per_row': float('nan'),
    })

    # Regression
    drop_for_reg = [c for c in TARGET_COLS if c != 'log_net_charge']
    ag_predictor_reg = TabularPredictor(
        label='log_net_charge', eval_metric='mean_absolute_error',
        path=str(MODELS / 'autogluon_reg'),
    ).fit(
        ag_train.drop(columns=drop_for_reg),
        tuning_data=ag_val.drop(columns=drop_for_reg),
        time_limit=1800, presets='medium_quality',
    )
    log_pred = ag_predictor_reg.predict(ag_test.drop(columns=drop_for_reg + ['log_net_charge'])).values
    dol_pred = np.expm1(log_pred)
    reg_rows.append({
        'model': 'AutoGluon',
        'mae_dollars':  mean_absolute_error(y_test_dol, dol_pred),
        'rmse_dollars': root_mean_squared_error(y_test_dol, dol_pred),
        'r2':           r2_score(y_test_dol, dol_pred),
        'latency_ms_per_row': float('nan'),
    })
    print(cls_rows[-1]); print(reg_rows[-1])
else:
    print('AutoGluon skipped (RUN_AUTOGLUON=False).')

AutoGluon skipped (RUN_AUTOGLUON=False).


## Comparison table

In [8]:
cls_table = pd.DataFrame(cls_rows).set_index('model').round(4)
reg_table = pd.DataFrame(reg_rows).set_index('model').round(4)

print('=== Classification (test set) ===')
print(cls_table.to_string())
print()
print('=== Regression (test set, dollars) ===')
print(reg_table.to_string())

cls_table.to_csv(MODELS / 'phase_3_classification.csv')
reg_table.to_csv(MODELS / 'phase_3_regression.csv')

=== Classification (test set) ===
            accuracy  precision  recall      f1     auc  latency_ms_per_row
model                                                                      
XGBoost v2    0.9974     0.9956  0.9962  0.9959  0.9997              0.0040
LightGBM      0.9972     0.9945  0.9967  0.9956  0.9999              0.0025
CatBoost      0.9972     0.9951  0.9962  0.9956  0.9994              0.0028

=== Regression (test set, dollars) ===
            mae_dollars  rmse_dollars      r2  latency_ms_per_row
model                                                            
XGBoost v2       2.7720        6.9323  0.8884              0.0061
LightGBM         2.7461        6.9543  0.8877              0.0406
CatBoost         2.6765        6.5744  0.8996              0.0024


## Pick the winner and persist as `best_v3`

Decision rules:
- **Classification**: pick highest AUC; tie-break with F1.
- **Regression**: pick lowest MAE.
- TabPFN's metrics use a 2K-row subsample, so we down-weight it unless it leads by >10%.
- Compare on per-row latency too: if a model wins by < 1% but is 10× slower, prefer the faster one.

In [9]:
# Down-weight TabPFN to avoid a subsampled-metric apples-to-oranges win
def adjusted_auc(row):
    return row['auc'] - (0.05 if row.name == 'TabPFN-2.5' else 0)
def adjusted_mae(row):
    return row['mae_dollars'] + (1.0 if row.name == 'TabPFN-2.5' else 0)

cls_winner = cls_table.assign(adj=cls_table.apply(adjusted_auc, axis=1)) \
                       .sort_values('adj', ascending=False).index[0]
reg_winner = reg_table.assign(adj=reg_table.apply(adjusted_mae, axis=1)) \
                       .sort_values('adj', ascending=True).index[0]

print(f'Classification winner: {cls_winner}')
print(f'Regression winner:     {reg_winner}')

model_lookup_cls = {
    'XGBoost v2': xgb_clf,
    'LightGBM':   lgb_clf,
    'CatBoost':   cat_clf,
}
model_lookup_reg = {
    'XGBoost v2': xgb_reg,
    'LightGBM':   lgb_reg,
    'CatBoost':   cat_reg,
}
if tab_clf is not None:
    model_lookup_cls['TabPFN-2.5'] = tab_clf
if tab_reg is not None:
    model_lookup_reg['TabPFN-2.5'] = tab_reg

best_cls = model_lookup_cls.get(cls_winner)
best_reg = model_lookup_reg.get(reg_winner)

if cls_winner == 'TabPFN-2.5':
    print('TabPFN won classification but is too slow for production; saving the runner-up instead.')
    runner = cls_table.drop('TabPFN-2.5').sort_values('auc', ascending=False).index[0]
    best_cls = model_lookup_cls[runner]
    cls_winner = f'{runner} (TabPFN edged it on AUC but latency disqualifies)'
if reg_winner == 'TabPFN-2.5':
    print('TabPFN won regression but is too slow for production; saving the runner-up instead.')
    runner = reg_table.drop('TabPFN-2.5').sort_values('mae_dollars').index[0]
    best_reg = model_lookup_reg[runner]
    reg_winner = f'{runner} (TabPFN edged it on MAE but latency disqualifies)'

joblib.dump(best_cls, MODELS / 'best_v3_classifier.pkl')
joblib.dump(best_reg, MODELS / 'best_v3_regressor.pkl')
joblib.dump({'classifier': best_cls, 'regressor': best_reg}, MODELS / 'best_v3.pkl')

metrics = {
    'classification_winner': cls_winner,
    'regression_winner':     reg_winner,
    'classification_table':  cls_table.reset_index().to_dict(orient='records'),
    'regression_table':      reg_table.reset_index().to_dict(orient='records'),
    'tabpfn_test_sample':    TABPFN_TEST_SAMPLE if (tab_clf is not None or tab_reg is not None) else None,
    'tabpfn_attempted':      TABPFN_OK,
}
with open(MODELS / 'phase_3_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2, default=str)

print('Saved models/best_v3.pkl and models/phase_3_metrics.json')

Classification winner: LightGBM
Regression winner:     CatBoost


Saved models/best_v3.pkl and models/phase_3_metrics.json


## Decision

The winning model from the cells above is persisted as `models/best_v3.pkl`. Read the comparison tables to see margins.

If XGBoost retained its lead this phase is mainly a confirmation result — that's still a finding, per the working agreement.
If a different model won, the next step is to (a) wire the new winner into `src/predict.py`, and (b) refit the Phase 2 conformal wrapper / isotonic calibrator on top of it before shipping.